author: loua

# Preprocess citizenship data for immigration research question

Goal: Create one CSV for each of the years in {2009, 2013, 2017, 2020}, specifically for immigration analyses. The CSVs should contain Citizenship data but where it's filtered with only ages 20+ years.



In [1]:
# Libraries
import pandas as pd
import numpy as np
import combine_raw_data as cb

# Configuration
from combine_raw_data import ROOT, PATH, COMMON_COLS, PREFIX, CPH_AREAS
YEARS = [2009, 2013, 2017, 2021]
OUTPUT_PATH = ROOT + 'processed-data/migration/'

In [2]:
def preprocess_citizenship_20years(year):
    csv_path = PATH + 'PersonsByCitizenshipSexAge.csv'
    pattern = f'{PREFIX}{year} - Antal personer opgjort efter statsborgerskab køn og aldersgrupper_'
    
    df = pd.read_csv(csv_path, sep=';', dtype=str)
    df = df[df['Gruppe'].isin(CPH_AREAS)]

    # column structure is '{pattern}{sex} {age_group}_{category number} {citizenship}' (with space) OR '{pattern}{sex} {age_group}_{category number}{citizenship}' (no space)
    # only use +20 years age groups
    sexes = [('Mænd', 'Male'), ('Kvinder', 'Female')]
    age_groups = ['20-24 år', '25-29 år', '30-34 år', '35-39 år', '40-44 år',
                  '45-49 år', '50-54 år', '55-59 år', '60-64 år', '65-69 år', '70- år']
    citizenships = [
        '01. Danmark',
        '02. Nordiske lande',
        '03. Tyrkiet',
        '04. Tidligere Jugoslavien',
        '05. Gamle EU-lande',
        '06. Nye EU-lande',
        '07. Øvrige Europa',
        '08. Afrika',
        '09. Nordamerika',
        '10. Syd- og Mellemamerika',
        '11. Asien og Oceanien',
        '12. Uoplyst'
    ]
    translations = {
        'Danmark': 'Denmark',
        'Nordiske lande': 'Nordic',
        'Tyrkiet': 'Turkey',
        'Tidligere Jugoslavien': 'Former Yugoslavia',
        'Gamle EU-lande': 'Old EU countries',
        'Nye EU-lande': 'New EU countries',
        'Øvrige Europa': 'Other Europe',
        'Afrika': 'Africa',
        'Nordamerika': 'North America',
        'Syd- og Mellemamerika': 'South and Central America',
        'Asien og Oceanien': 'Asia and Oceania',
        'Uoplyst': 'Not specified'
    }

    # helper function to clean numeric columns, treat '-' as NaN
    def clean_column(col_name):
        if col_name in df.columns:
            target_col = col_name
        else:
            # Try to find with stripped whitespace
            matching = [c for c in df.columns if c.strip() == col_name.strip()]
            if not matching:
                return None
            target_col = matching[0]
        col_data = df[target_col].astype(str).str.strip()
        col_data = col_data.str.replace(',', '.')
        col_data = col_data.replace('-', '')  # Replace '-' with empty string instead of NaN
        return pd.to_numeric(col_data, errors='coerce')  # Empty strings become NaN automatically

    # helper function to deal with NaN/0 - if all vals in a col are null(0), return NaN series, otherwise replace NaN with 0
    def finalise_column(series):
        if (series.fillna(0) == 0).all():
            return pd.Series(np.nan, index=series.index)
        return series.fillna(0)
    
    result_df = df[COMMON_COLS].copy()

    for citizenship in citizenships:
        citizenship_sum = pd.Series(0.0, index=df.index)
        citizenship_found = False
            
        for age in age_groups:
            for sex in sexes:
                col = f'{pattern}{sex[0]} {age}_{citizenship}'
                vals = clean_column(col)
                if vals is not None:
                    citizenship_sum += vals.fillna(0)
                    citizenship_found = True
        
        if citizenship_found:
            citizenship_name = citizenship.split('. ')[1]
            result_df[f'Citizenship_{translations[citizenship_name]}'] = finalise_column(citizenship_sum)
        else:
            citizenship_name = citizenship[0].split('. ')[1]
            result_df[f'Citizenship_{translations[citizenship_name]}'] = np.nan
    
    print(f"    Citizenship data: Using with-space format for year {year}")
    result_df['Citizenship_Total'] = result_df.filter(like='Citizenship_').sum(axis=1)
    return result_df

In [3]:
df_2009 = preprocess_citizenship_20years(2009)
df_2013 = preprocess_citizenship_20years(2013)
df_2017 = preprocess_citizenship_20years(2017)
df_2021 = preprocess_citizenship_20years(2021)

DFS = [df_2009, df_2013, df_2017, df_2021]

    Citizenship data: Using with-space format for year 2009
    Citizenship data: Using with-space format for year 2013
    Citizenship data: Using with-space format for year 2017
    Citizenship data: Using with-space format for year 2021


In [4]:
df_2021.head()

,Gruppe,ValgstedId,KredsNr,StorKredsNr,LandsdelsNr,Citizenship_Denmark,Citizenship_Nordic,Citizenship_Turkey,Citizenship_Former Yugoslavia,Citizenship_Old EU countries,Citizenship_New EU countries,Citizenship_Other Europe,Citizenship_Africa,Citizenship_North America,Citizenship_South and Central America,Citizenship_Asia and Oceania,Citizenship_Not specified,Citizenship_Total
0,101001,101001,1,1,1,10754.0,329.0,12.0,32.0,803.0,185.0,42.0,50.0,123.0,80.0,275.0,6.0,12691.0
1,101002,101002,1,1,1,5798.0,133.0,18.0,25.0,326.0,96.0,35.0,37.0,42.0,47.0,236.0,4.0,6797.0
2,101003,101003,1,1,1,10710.0,313.0,18.0,17.0,684.0,127.0,36.0,32.0,101.0,144.0,266.0,1.0,12449.0
3,101005,101005,1,1,1,10589.0,262.0,45.0,113.0,778.0,274.0,80.0,94.0,164.0,126.0,606.0,3.0,13134.0
4,101006,101006,1,1,1,9302.0,213.0,34.0,75.0,517.0,225.0,59.0,98.0,62.0,68.0,328.0,9.0,10990.0


In [5]:
# Compare with the one with all ages
citizenship_2021 = cb.preprocess_citizenship_sex_age(2021)
citizenship_2021.head()

    Citizenship data: Using with-space format for year 2021


,Gruppe,ValgstedId,KredsNr,StorKredsNr,LandsdelsNr,Sex_Male,Sex_Female,Age_0-4 years,Age_5-9 years,Age_10-14 years,...,Citizenship_Former Yugoslavia,Citizenship_Old EU countries,Citizenship_New EU countries,Citizenship_Other Europe,Citizenship_Africa,Citizenship_North America,Citizenship_South and Central America,Citizenship_Asia and Oceania,Citizenship_Not specified,Citizenship_Total
0,101001,101001,1,1,1,7331.0,8231.0,853.0,657.0,678.0,...,39.0,934.0,202.0,46.0,62.0,147.0,87.0,325.0,8.0,15562.0
1,101002,101002,1,1,1,3908.0,4470.0,426.0,383.0,397.0,...,27.0,372.0,106.0,41.0,45.0,52.0,49.0,271.0,6.0,8378.0
2,101003,101003,1,1,1,7640.0,8206.0,920.0,771.0,862.0,...,19.0,763.0,143.0,46.0,39.0,120.0,147.0,309.0,1.0,15846.0
3,101005,101005,1,1,1,7648.0,8206.0,929.0,639.0,593.0,...,133.0,827.0,297.0,95.0,110.0,178.0,132.0,677.0,3.0,15854.0
4,101006,101006,1,1,1,6527.0,6960.0,801.0,563.0,570.0,...,82.0,575.0,252.0,67.0,126.0,67.0,70.0,406.0,10.0,13487.0


## Drop null columns

In [6]:
print(f"\n{'='*80}\n  2009\n")
print(df_2009.info(verbose=True, show_counts=True))
print(f"\n{'='*80}\n  2013\n")
print(df_2013.info(verbose=True, show_counts=True))
print(f"\n{'='*80}\n  2017\n")
print(df_2017.info(verbose=True, show_counts=True))
print(f"\n{'='*80}\n  2021\n")
print(df_2021.info(verbose=True, show_counts=True))


  2009

<class 'pandas.core.frame.DataFrame'>
Index: 58 entries, 0 to 57
Data columns (total 18 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Gruppe                                 58 non-null     object 
 1   ValgstedId                             58 non-null     object 
 2   KredsNr                                58 non-null     object 
 3   StorKredsNr                            58 non-null     object 
 4   LandsdelsNr                            58 non-null     object 
 5   Citizenship_Denmark                    58 non-null     float64
 6   Citizenship_Nordic                     58 non-null     float64
 7   Citizenship_Turkey                     58 non-null     float64
 8   Citizenship_Former Yugoslavia          58 non-null     float64
 9   Citizenship_Old EU countries           58 non-null     float64
 10  Citizenship_New EU countries           58 non-null     float64
 11  Citi

In [7]:
for i, year in enumerate(YEARS):
    DFS[i] = DFS[i].dropna(axis=1, how='all')

df_2009, df_2013, df_2017, df_2021 = DFS

In [8]:
print(f"\n{'='*80}\n  2009\n")
print(df_2009.info(verbose=True, show_counts=True))
print(f"\n{'='*80}\n  2013\n")
print(df_2013.info(verbose=True, show_counts=True))
print(f"\n{'='*80}\n  2017\n")
print(df_2017.info(verbose=True, show_counts=True))
print(f"\n{'='*80}\n  2021\n")
print(df_2021.info(verbose=True, show_counts=True))


  2009

<class 'pandas.core.frame.DataFrame'>
Index: 58 entries, 0 to 57
Data columns (total 17 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Gruppe                                 58 non-null     object 
 1   ValgstedId                             58 non-null     object 
 2   KredsNr                                58 non-null     object 
 3   StorKredsNr                            58 non-null     object 
 4   LandsdelsNr                            58 non-null     object 
 5   Citizenship_Denmark                    58 non-null     float64
 6   Citizenship_Nordic                     58 non-null     float64
 7   Citizenship_Turkey                     58 non-null     float64
 8   Citizenship_Former Yugoslavia          58 non-null     float64
 9   Citizenship_Old EU countries           58 non-null     float64
 10  Citizenship_New EU countries           58 non-null     float64
 11  Citi

## Add and drop geography columns

In [9]:
def add_geography_columns(df, geography_csv_path):
    """
    Add geography columns to dataframe by matching ValgstedId.
    
    Parameters:
    - df: DataFrame to add columns to
    - geography_csv_path: Path to Geography.csv
    
    Returns:
    - DataFrame with added geography columns
    """    
    # Read Geography.csv
    geo_df = pd.read_csv(geography_csv_path, sep=';', encoding='utf-8', quotechar='"')
    
    # Clean column names (remove any extra whitespace/quotes)
    geo_df.columns = geo_df.columns.str.strip().str.strip('"')

    # Filter for KommuneNr 101 or 147
    geo_df = geo_df[geo_df['KommuneNr'].astype(str).isin(['101', '147'])].copy()

    # Debug: print first few rows to see what we're working with
    print("Geography CSV columns:", geo_df.columns.tolist())
    print("First few Valgsted Ids:", geo_df['Valgsted Id'].head().tolist())

    # Create a dictionary for quick lookup
    geo_dict = {}
    for _, row in geo_df.iterrows():
        # Clean the ValgstedId (remove quotes, whitespace)
        valgsted_id = str(row['Valgsted Id']).strip().strip('"')
        geo_dict[valgsted_id] = {
            'Valgsted navn': str(row['Valgsted navn']).strip().strip('"'),
            'Kreds navn': str(row['Kreds navn']).strip().strip('"'),
            'Kommune navn': str(row['Kommune navn']).strip().strip('"')
        }
    
    print(f"Total entries in geo_dict: {len(geo_dict)}")
    print(f"Sample keys: {list(geo_dict.keys())[:5]}")
    
    # Create new columns
    df_result = df.copy()
    valgsted_navne = []
    kreds_navne = []
    kommune_navne = []
    
    for _, row in df_result.iterrows():
        valgsted_ids = str(row['ValgstedId']).split(';')
        
        # Collect names for each ValgstedId
        v_names = []
        k_names = []
        kom_names = []
        
        for vid in valgsted_ids:
            vid = vid.strip()
            if vid not in geo_dict:
                print(f"Missing ValgstedId: '{vid}' (length: {len(vid)})")
                print(f"Available keys starting with '101': {[k for k in geo_dict.keys() if k.startswith('101')][:10]}")
                raise ValueError(f"ValgstedId '{vid}' not found in Geography.csv")
            
            v_names.append(geo_dict[vid]['Valgsted navn'])
            k_names.append(geo_dict[vid]['Kreds navn'])
            kom_names.append(geo_dict[vid]['Kommune navn'])
        
        # Join with semicolons, remove duplicates while preserving order
        valgsted_navne.append(';'.join(v_names))
        
        # For Kreds and Kommune, use single value if all same, otherwise semicolon-separate
        if len(set(k_names)) == 1:
            kreds_navne.append(k_names[0])
        else:
            kreds_navne.append(';'.join(k_names))
            
        if len(set(kom_names)) == 1:
            kommune_navne.append(kom_names[0])
        else:
            kommune_navne.append(';'.join(kom_names))
    
    # Add the new columns
    df_result['Valgsted navn'] = valgsted_navne
    df_result['Kreds navn'] = kreds_navne
    df_result['Kommune navn'] = kommune_navne
    
    # Drop the columns we don't want
    df_result = df_result.drop(columns=['StorKredsNr', 'LandsdelsNr'])
    
    # Reorder columns to put the first 6 in the desired order
    first_cols = ['Gruppe', 'ValgstedId', 'Valgsted navn', 'KredsNr', 'Kreds navn', 'Kommune navn']
    other_cols = [col for col in df_result.columns if col not in first_cols]
    df_result = df_result[first_cols + other_cols]
    
    return df_result

In [10]:
geo_path = ROOT + 'raw-data/historical-elections/election-results/Geography.csv'

for i, year in enumerate(YEARS):
    DFS[i] = add_geography_columns(DFS[i], geo_path)

df_2009, df_2013, df_2017, df_2021 = DFS

# Check the result
print("\nFirst few rows of df_2021:")
print(df_2021[['Gruppe', 'ValgstedId', 'Valgsted navn', 'KredsNr', 'Kreds navn', 'Kommune navn']].head(10))

Geography CSV columns: ['Valgsted Id', 'KommuneNr', 'Kreds Nr', 'Storkreds Nr', 'Landsdels Nr', 'Valgsted navn', 'Kommune navn', 'Kreds navn', 'Storkreds navn', 'Landsdels navn', 'Valgsted start', 'Valgsted stop']
First few Valgsted Ids: [101001, 101002, 101003, 101005, 101006]
Total entries in geo_dict: 61
Sample keys: ['101001', '101002', '101003', '101005', '101006']
Geography CSV columns: ['Valgsted Id', 'KommuneNr', 'Kreds Nr', 'Storkreds Nr', 'Landsdels Nr', 'Valgsted navn', 'Kommune navn', 'Kreds navn', 'Storkreds navn', 'Landsdels navn', 'Valgsted start', 'Valgsted stop']
First few Valgsted Ids: [101001, 101002, 101003, 101005, 101006]
Total entries in geo_dict: 61
Sample keys: ['101001', '101002', '101003', '101005', '101006']
Geography CSV columns: ['Valgsted Id', 'KommuneNr', 'Kreds Nr', 'Storkreds Nr', 'Landsdels Nr', 'Valgsted navn', 'Kommune navn', 'Kreds navn', 'Storkreds navn', 'Landsdels navn', 'Valgsted start', 'Valgsted stop']
First few Valgsted Ids: [101001, 101002,

In [11]:
df_2013.head()

,Gruppe,ValgstedId,Valgsted navn,KredsNr,Kreds navn,Kommune navn,Citizenship_Denmark,Citizenship_Nordic,Citizenship_Turkey,Citizenship_Former Yugoslavia,Citizenship_Old EU countries,Citizenship_New EU countries,Citizenship_Other Europe,Citizenship_Africa,Citizenship_North America,Citizenship_South and Central America,Citizenship_Asia and Oceania,Citizenship_Not specified,Citizenship_Total
0,101001,101001,1. Østerbro,1,1. Østerbro,København,10108.61,267.37,17.71,23.10,466.18,104.01,41.58,51.59,100.92,44.66,223.47,1.54,11450.74
1,101002,101002,1. Nord,1,1. Østerbro,København,5573.00,120.00,16.00,20.00,281.00,99.00,35.00,26.00,41.00,27.00,205.00,4.00,6447.00
2,101003,101003,1. Syd,1,1. Østerbro,København,11177.56,355.37,14.54,29.04,601.17,120.89,48.38,54.80,121.69,55.62,285.28,3.24,12867.58
3,101005,101005,1. Vest,1,1. Østerbro,København,10607.00,264.00,51.00,94.00,609.00,224.00,68.00,92.00,143.00,64.00,442.00,5.00,12663.00
4,101006,101006,1. Nordvest,1,1. Østerbro,København,9279.00,185.00,41.00,66.00,349.00,206.00,60.00,90.00,71.00,43.00,319.00,8.00,10717.00


## Convert (some) floats to ints

In [12]:
def convert_floats_to_int(df):
    """Convert float columns to int if they have no fractional parts"""
    df_converted = df.copy()
    
    for col in df_converted.columns:
        # Check if column is float type
        if df_converted[col].dtype == 'float64':
            # Check if all non-null values have no fractional part
            non_null_values = df_converted[col].dropna()
            if len(non_null_values) > 0 and (non_null_values % 1 == 0).all():
                # Convert to Int64 (nullable integer type to preserve NaN if present)
                df_converted[col] = df_converted[col].astype('int64')
    
    return df_converted

In [13]:
for i, year in enumerate(YEARS):
    DFS[i] = convert_floats_to_int(DFS[i])

df_2009, df_2013, df_2017, df_2021 = DFS

In [14]:
df_2013.info() # should have floats

<class 'pandas.core.frame.DataFrame'>
Index: 58 entries, 0 to 57
Data columns (total 19 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Gruppe                                 58 non-null     object 
 1   ValgstedId                             58 non-null     object 
 2   Valgsted navn                          58 non-null     object 
 3   KredsNr                                58 non-null     object 
 4   Kreds navn                             58 non-null     object 
 5   Kommune navn                           58 non-null     object 
 6   Citizenship_Denmark                    58 non-null     float64
 7   Citizenship_Nordic                     58 non-null     float64
 8   Citizenship_Turkey                     58 non-null     float64
 9   Citizenship_Former Yugoslavia          58 non-null     float64
 10  Citizenship_Old EU countries           58 non-null     float64
 11  Citizenship_N

In [15]:
df_2021.info() # should have ints

<class 'pandas.core.frame.DataFrame'>
Index: 58 entries, 0 to 57
Data columns (total 19 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   Gruppe                                 58 non-null     object
 1   ValgstedId                             58 non-null     object
 2   Valgsted navn                          58 non-null     object
 3   KredsNr                                58 non-null     object
 4   Kreds navn                             58 non-null     object
 5   Kommune navn                           58 non-null     object
 6   Citizenship_Denmark                    58 non-null     int64 
 7   Citizenship_Nordic                     58 non-null     int64 
 8   Citizenship_Turkey                     58 non-null     int64 
 9   Citizenship_Former Yugoslavia          58 non-null     int64 
 10  Citizenship_Old EU countries           58 non-null     int64 
 11  Citizenship_New EU countri

## Save to CSVs

In [16]:
for i, year in enumerate(YEARS):
    filename = f'{OUTPUT_PATH}absolute_citizenship_over20years_{year}.csv'
    DFS[i].to_csv(filename, index=False)
    print(f"Saved {filename}")

Saved ../../../processed-data/migration/absolute_citizenship_over20years_2009.csv
Saved ../../../processed-data/migration/absolute_citizenship_over20years_2013.csv
Saved ../../../processed-data/migration/absolute_citizenship_over20years_2017.csv
Saved ../../../processed-data/migration/absolute_citizenship_over20years_2021.csv


## Convert to percentages and save to new CSVs

In [17]:
for i, year in enumerate(YEARS):
    df = DFS[i].copy()
    citizenship_cols = [col for col in df.columns 
                       if col.startswith('Citizenship_') and col != 'Citizenship_Total']
    df[citizenship_cols] = df[citizenship_cols].div(df['Citizenship_Total'], axis=0) * 100
    filename = f'{OUTPUT_PATH}relative_citizenship_over20years_{year}.csv'
    df.to_csv(filename, index=False)
    print(f"Saved {filename}")

Saved ../../../processed-data/migration/relative_citizenship_over20years_2009.csv
Saved ../../../processed-data/migration/relative_citizenship_over20years_2013.csv
Saved ../../../processed-data/migration/relative_citizenship_over20years_2017.csv
Saved ../../../processed-data/migration/relative_citizenship_over20years_2021.csv
